# 🔧 Recovery: Add article_id to Remaining (For Prediction)

**Problem:** Your `remaining_for_prediction.csv` doesn't have article_id for tracking!

**Solution:** Add article_id to remaining data (articles NOT in labeled seed)

**Result:** Full traceability for all articles (labeled + remaining)!

---

## 📦 Step 1: Setup

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries loaded!")
print(f"\n⚠️ IMPORTANT: Make sure you have:")
print(f"   1. cnbc_cleaned_full.csv (OR cnbc_cleaned_full_WITH_ID.csv)")
print(f"   2. cnbc_sampled_article_ids.txt (from recovery notebook)")
print(f"   3. cnbc_remaining_for_prediction.csv (original remaining file)")

✅ Libraries loaded!

⚠️ IMPORTANT: Make sure you have:
   1. cnbc_cleaned_full.csv (OR cnbc_cleaned_full_WITH_ID.csv)
   2. cnbc_sampled_article_ids.txt (from recovery notebook)
   3. cnbc_remaining_for_prediction.csv (original remaining file)


## 📥 Step 2: Load Full Cleaned Data (with article_id)

In [3]:
print("="*70)
print("📥 STEP 2: LOADING FULL CLEANED DATA")
print("="*70)

# Try to load WITH_ID version first (if recovery was run)
try:
    df_clean = pd.read_csv('CNBC_hasil/cnbc_cleaned_full_WITH_ID.csv', encoding='utf-8-sig')
    print(f"✅ Loaded: cnbc_cleaned_full_WITH_ID.csv")
    has_id = 'article_id' in df_clean.columns
except:
    # If not, load original and add article_id
    df_clean = pd.read_csv('cnbc_cleaned_full.csv', encoding='utf-8-sig')
    print(f"✅ Loaded: cnbc_cleaned_full.csv")
    has_id = False

print(f"\n📊 Dataset info:")
print(f"   Total articles: {len(df_clean):,}")
print(f"   Columns: {df_clean.columns.tolist()}")

# Add article_id if not present
if not has_id:
    print(f"\n🆔 Adding article_id...")
    df_clean['article_id'] = 'CNBC_' + df_clean.index.astype(str).str.zfill(5)
    print(f"   ✅ Added article_id: {df_clean['article_id'].iloc[0]} to {df_clean['article_id'].iloc[-1]}")
else:
    print(f"\n✅ article_id already present!")

print(f"\nSample data:")
display(df_clean[['article_id', 'date', 'title']].head(3))

📥 STEP 2: LOADING FULL CLEANED DATA
✅ Loaded: cnbc_cleaned_full_WITH_ID.csv

📊 Dataset info:
   Total articles: 5,060
   Columns: ['date', 'title', 'content', 'content_length', 'article_id']

✅ article_id already present!

Sample data:


,article_id,date,title
0,CNBC_00000,2019-09-01,"Kondusif, Menkominfo Siap Buka Kembali Internet di Papua"
1,CNBC_00001,2019-09-02,Wiranto Pastikan Papua Mulai Kondusif
2,CNBC_00002,2019-09-03,"Brexit Berlarut-larut, Boris Johnson Ancam Parlemen Inggris"


## 📋 Step 3: Load Sampled IDs (Articles Already Labeled)

In [4]:
print("="*70)
print("📋 STEP 3: LOADING SAMPLED IDs")
print("="*70)

# Load list of article_ids that were sampled (labeled)
try:
    with open('CNBC_hasil/cnbc_sampled_article_ids.txt', 'r') as f:
        sampled_ids = [line.strip() for line in f.readlines() if line.strip()]
    print(f"✅ Loaded sampled IDs from: cnbc_sampled_article_ids.txt")
    print(f"   Total sampled (labeled): {len(sampled_ids)} articles")
except FileNotFoundError:
    print(f"\n⚠️ WARNING: cnbc_sampled_article_ids.txt not found!")
    print(f"   This file is created by the Recovery_Add_Article_ID notebook")
    print(f"   Please run that notebook first!")
    print(f"\n💡 Alternative: Recreate sampling with seed=42")
    
    # Recreate sampling
    print(f"\n🎲 Recreating sampling (seed=42)...")
    np.random.seed(42)
    df_sample = df_clean.sample(n=800, random_state=42)
    sampled_ids = df_sample['article_id'].tolist()
    print(f"   ✅ Recreated: {len(sampled_ids)} sampled IDs")
    
    # Save for future use
    with open('cnbc_sampled_article_ids.txt', 'w') as f:
        f.write('\n'.join(sampled_ids))
    print(f"   ✅ Saved to: cnbc_sampled_article_ids.txt")

print(f"\nFirst 5 sampled IDs:")
for i, article_id in enumerate(sampled_ids[:5], 1):
    print(f"   {i}. {article_id}")

📋 STEP 3: LOADING SAMPLED IDs
✅ Loaded sampled IDs from: cnbc_sampled_article_ids.txt
   Total sampled (labeled): 799 articles

First 5 sampled IDs:
   1. CNBC_00008
   2. CNBC_00012
   3. CNBC_00017
   4. CNBC_00023
   5. CNBC_00026


## 🔍 Step 4: Extract Remaining Articles (NOT Labeled)

In [5]:
print("="*70)
print("🔍 STEP 4: EXTRACTING REMAINING ARTICLES")
print("="*70)

# Get remaining articles (NOT in sampled_ids)
df_remaining = df_clean[~df_clean['article_id'].isin(sampled_ids)].copy()
df_remaining = df_remaining.sort_values('date').reset_index(drop=True)

print(f"\n📊 Split Summary:")
print(f"   Total articles: {len(df_clean):,}")
print(f"   Sampled (labeled): {len(sampled_ids):,}")
print(f"   Remaining (for prediction): {len(df_remaining):,}")
print(f"   Check: {len(sampled_ids) + len(df_remaining)} = {len(df_clean)} ✅")

# Verify no overlap
overlap = df_remaining['article_id'].isin(sampled_ids).sum()
if overlap == 0:
    print(f"\n✅ NO OVERLAP! Remaining articles are truly unlabeled!")
else:
    print(f"\n⚠️ WARNING: {overlap} articles overlap with sampled!")
    print(f"   This shouldn't happen - check your data!")

print(f"\n📅 Date Range - Remaining:")
print(f"   From: {df_remaining['date'].min()}")
print(f"   To: {df_remaining['date'].max()}")

print(f"\nSample remaining articles:")
display(df_remaining[['article_id', 'date', 'title']].head(5))

🔍 STEP 4: EXTRACTING REMAINING ARTICLES

📊 Split Summary:
   Total articles: 5,060
   Sampled (labeled): 799
   Remaining (for prediction): 4,261
   Check: 5060 = 5060 ✅

✅ NO OVERLAP! Remaining articles are truly unlabeled!

📅 Date Range - Remaining:
   From: 2019-09-01
   To: 2024-09-30

Sample remaining articles:


,article_id,date,title
0,CNBC_00000,2019-09-01,"Kondusif, Menkominfo Siap Buka Kembali Internet di Papua"
1,CNBC_00001,2019-09-02,Wiranto Pastikan Papua Mulai Kondusif
2,CNBC_00002,2019-09-03,"Brexit Berlarut-larut, Boris Johnson Ancam Parlemen Inggris"
3,CNBC_00003,2019-09-04,"Salip RI, di Vietnam Tanah Gratis, Buruhnya Produktif"
4,CNBC_00004,2019-09-04,Pemimpin Hong Kong Resmi Cabut RUU Ekstradisi


## 🔄 Step 5: Load & Compare with Old Remaining File

In [6]:
print("="*70)
print("🔄 STEP 5: COMPARING WITH OLD REMAINING FILE")
print("="*70)

# Try to load old remaining file (if exists)
try:
    df_old_remaining = pd.read_csv('CNBC_hasil/cnbc_remaining_for_prediction.csv', encoding='utf-8-sig')
    print(f"✅ Loaded old file: cnbc_remaining_for_prediction.csv")
    print(f"   Rows in old file: {len(df_old_remaining):,}")
    print(f"   Columns: {df_old_remaining.columns.tolist()}")
    
    # Compare counts
    print(f"\n📊 Comparison:")
    print(f"   Old remaining file: {len(df_old_remaining):,} articles")
    print(f"   New remaining file: {len(df_remaining):,} articles")
    
    if len(df_old_remaining) == len(df_remaining):
        print(f"   ✅ MATCH! Same number of articles!")
    else:
        diff = abs(len(df_old_remaining) - len(df_remaining))
        print(f"   ⚠️ DIFFERENCE: {diff} articles")
        print(f"   This is normal if files were created at different times")
    
    # Check if old file has article_id
    if 'article_id' in df_old_remaining.columns:
        print(f"\n✅ Old file already has article_id!")
        print(f"   No need to update - use existing file")
    else:
        print(f"\n⚠️ Old file MISSING article_id!")
        print(f"   Will create new file with article_id")
    
except FileNotFoundError:
    print(f"ℹ️ Old remaining file not found (cnbc_remaining_for_prediction.csv)")
    print(f"   This is OK - will create new one with article_id")

🔄 STEP 5: COMPARING WITH OLD REMAINING FILE
✅ Loaded old file: cnbc_remaining_for_prediction.csv
   Rows in old file: 4,260
   Columns: ['date', 'title', 'content', 'content_length']

📊 Comparison:
   Old remaining file: 4,260 articles
   New remaining file: 4,261 articles
   ⚠️ DIFFERENCE: 1 articles
   This is normal if files were created at different times

⚠️ Old file MISSING article_id!
   Will create new file with article_id


## 💾 Step 6: Save Remaining with article_id

In [7]:
print("="*70)
print("💾 STEP 6: SAVING REMAINING WITH article_id")
print("="*70)

# Select columns for output
output_columns = ['article_id', 'date', 'title', 'content', 'content_length']

# Make sure all columns exist
available_columns = [col for col in output_columns if col in df_remaining.columns]
df_output = df_remaining[available_columns].copy()

print(f"\n📋 Output columns: {available_columns}")

# Save main file
output_file = 'cnbc_remaining_for_prediction_WITH_ID.csv'
df_output.to_csv(output_file, index=False, encoding='utf-8-sig')
file_size = os.path.getsize(output_file) / (1024**2)

print(f"\n1️⃣ Saved: {output_file}")
print(f"   Rows: {len(df_output):,}")
print(f"   Size: {file_size:.2f} MB")
print(f"   Purpose: Articles to be predicted by model (with article_id!)")

# Save list of remaining IDs
remaining_ids = df_output['article_id'].tolist()
id_file = 'cnbc_remaining_article_ids.txt'
with open(id_file, 'w') as f:
    f.write('\n'.join(remaining_ids))

print(f"\n2️⃣ Saved: {id_file}")
print(f"   Total IDs: {len(remaining_ids):,}")
print(f"   Purpose: Track which articles are for prediction")

# Also update the cleaned_full file if needed
if not has_id:
    clean_with_id_file = 'cnbc_cleaned_full_WITH_ID.csv'
    df_clean.to_csv(clean_with_id_file, index=False, encoding='utf-8-sig')
    file_size = os.path.getsize(clean_with_id_file) / (1024**2)
    print(f"\n3️⃣ Saved: {clean_with_id_file}")
    print(f"   Rows: {len(df_clean):,}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   Purpose: Full dataset with article_id (for future use)")

print(f"\n{'='*70}")
print(f"✅ ALL FILES SAVED!")
print(f"{'='*70}")

💾 STEP 6: SAVING REMAINING WITH article_id

📋 Output columns: ['article_id', 'date', 'title', 'content', 'content_length']

1️⃣ Saved: cnbc_remaining_for_prediction_WITH_ID.csv
   Rows: 4,261
   Size: 13.33 MB
   Purpose: Articles to be predicted by model (with article_id!)

2️⃣ Saved: cnbc_remaining_article_ids.txt
   Total IDs: 4,261
   Purpose: Track which articles are for prediction

✅ ALL FILES SAVED!


## ✅ Step 7: Validation - Verify Split

In [8]:
print("="*70)
print("✅ STEP 7: VALIDATION - Verify Split")
print("="*70)

# Load both files to verify
print(f"\n📋 Loading files for verification...")
try:
    df_labeled = pd.read_csv('CNBC_hasil/cnbc_labeled_WITH_ID.csv', encoding='utf-8-sig')
    labeled_ids = set(df_labeled['article_id'].tolist())
    print(f"   ✅ Labeled file: {len(labeled_ids)} articles")
except:
    print(f"   ⚠️ Labeled file not found (run Recovery_Add_Article_ID first)")
    labeled_ids = set(sampled_ids)
    print(f"   Using sampled_ids: {len(labeled_ids)} articles")

df_remaining_check = pd.read_csv(output_file, encoding='utf-8-sig')
remaining_ids = set(df_remaining_check['article_id'].tolist())
print(f"   ✅ Remaining file: {len(remaining_ids)} articles")

# Check 1: Total count
print(f"\n1️⃣ Total Count Check:")
total = len(labeled_ids) + len(remaining_ids)
print(f"   Labeled: {len(labeled_ids):,}")
print(f"   Remaining: {len(remaining_ids):,}")
print(f"   Total: {total:,}")
print(f"   Expected: {len(df_clean):,}")
if total == len(df_clean):
    print(f"   ✅ PERFECT! All articles accounted for!")
else:
    print(f"   ⚠️ Mismatch: {abs(total - len(df_clean))} articles difference")

# Check 2: No overlap
print(f"\n2️⃣ Overlap Check:")
overlap = labeled_ids & remaining_ids
print(f"   Overlap: {len(overlap)} articles")
if len(overlap) == 0:
    print(f"   ✅ PERFECT! No overlap between labeled and remaining!")
else:
    print(f"   ❌ ERROR: {len(overlap)} articles in BOTH labeled and remaining!")
    print(f"   Overlapping IDs:")
    for article_id in list(overlap)[:5]:
        print(f"      - {article_id}")

# Check 3: No duplicates
print(f"\n3️⃣ Duplicate Check:")
labeled_dups = len(df_labeled['article_id']) - len(labeled_ids)
remaining_dups = len(df_remaining_check['article_id']) - len(remaining_ids)
print(f"   Duplicates in labeled: {labeled_dups}")
print(f"   Duplicates in remaining: {remaining_dups}")
if labeled_dups == 0 and remaining_dups == 0:
    print(f"   ✅ PERFECT! No duplicates!")
else:
    print(f"   ⚠️ WARNING: Duplicates found!")

# Check 4: All IDs valid format
print(f"\n4️⃣ ID Format Check:")
invalid_labeled = [id for id in labeled_ids if not id.startswith('CNBC_')]
invalid_remaining = [id for id in remaining_ids if not id.startswith('CNBC_')]
print(f"   Invalid IDs in labeled: {len(invalid_labeled)}")
print(f"   Invalid IDs in remaining: {len(invalid_remaining)}")
if len(invalid_labeled) == 0 and len(invalid_remaining) == 0:
    print(f"   ✅ PERFECT! All IDs have correct format (CNBC_XXXXX)")
else:
    print(f"   ⚠️ WARNING: Some IDs don't follow CNBC_XXXXX format")

print(f"\n{'='*70}")
print(f"VALIDATION SUMMARY")
print(f"{'='*70}")
if (total == len(df_clean) and len(overlap) == 0 and 
    labeled_dups == 0 and remaining_dups == 0):
    print(f"✅ ALL CHECKS PASSED!")
    print(f"   • Perfect split: {len(labeled_ids)} labeled + {len(remaining_ids)} remaining")
    print(f"   • No overlap between sets")
    print(f"   • No duplicates")
    print(f"   • All IDs valid format")
    print(f"\n🎉 READY FOR MODEL TRAINING & PREDICTION!")
else:
    print(f"⚠️ SOME ISSUES DETECTED - See checks above!")

✅ STEP 7: VALIDATION - Verify Split

📋 Loading files for verification...
   ✅ Labeled file: 799 articles
   ✅ Remaining file: 4261 articles

1️⃣ Total Count Check:
   Labeled: 799
   Remaining: 4,261
   Total: 5,060
   Expected: 5,060
   ✅ PERFECT! All articles accounted for!

2️⃣ Overlap Check:
   Overlap: 0 articles
   ✅ PERFECT! No overlap between labeled and remaining!

3️⃣ Duplicate Check:
   Duplicates in labeled: 0
   Duplicates in remaining: 0
   ✅ PERFECT! No duplicates!

4️⃣ ID Format Check:
   Invalid IDs in labeled: 0
   Invalid IDs in remaining: 0
   ✅ PERFECT! All IDs have correct format (CNBC_XXXXX)

VALIDATION SUMMARY
✅ ALL CHECKS PASSED!
   • Perfect split: 799 labeled + 4261 remaining
   • No overlap between sets
   • No duplicates
   • All IDs valid format

🎉 READY FOR MODEL TRAINING & PREDICTION!


## 📊 Step 8: Final Summary & Statistics

In [9]:
print("="*70)
print("📊 FINAL SUMMARY & STATISTICS")
print("="*70)

print(f"\n📁 FILES CREATED/UPDATED:")
print(f"\n1. cnbc_remaining_for_prediction_WITH_ID.csv")
print(f"   • {len(df_remaining):,} articles for prediction")
print(f"   • All have article_id for tracking")
print(f"   • Ready for model inference")

print(f"\n2. cnbc_remaining_article_ids.txt")
print(f"   • List of {len(remaining_ids):,} remaining IDs")
print(f"   • For tracking & reproducibility")

print(f"\n3. cnbc_cleaned_full_WITH_ID.csv (if created)")
print(f"   • Full dataset: {len(df_clean):,} articles")
print(f"   • All with article_id")
print(f"   • Master file for future use")

print(f"\n📊 DATASET SPLIT:")
print(f"\n┌─────────────────────────────────────────────┐")
print(f"│ Full Dataset: {len(df_clean):,} articles                │")
print(f"├─────────────────────────────────────────────┤")
print(f"│ ✅ Labeled (manual): {len(sampled_ids):,} articles       │")
print(f"│    → cnbc_labeled_WITH_ID.csv             │")
print(f"│    → For training model                   │")
print(f"├─────────────────────────────────────────────┤")
print(f"│ 🔮 Remaining (prediction): {len(df_remaining):,} articles │")
print(f"│    → cnbc_remaining_WITH_ID.csv           │")
print(f"│    → For model inference                  │")
print(f"└─────────────────────────────────────────────┘")

print(f"\n📅 DATE DISTRIBUTION:")
print(f"\nRemaining articles:")
print(f"   From: {df_remaining['date'].min()}")
print(f"   To: {df_remaining['date'].max()}")
print(f"   Days: {(pd.to_datetime(df_remaining['date'].max()) - pd.to_datetime(df_remaining['date'].min())).days}")

# Monthly distribution
df_remaining['year_month'] = pd.to_datetime(df_remaining['date']).dt.to_period('M')
monthly = df_remaining['year_month'].value_counts().sort_index()
print(f"\n   Articles per month (average): {monthly.mean():.1f}")

print(f"\n{'='*70}")
print(f"✅ RECOVERY COMPLETE - REMAINING DATA!")
print(f"{'='*70}")

print(f"\n🎯 FULL TRACEABILITY ACHIEVED!")
print(f"   ✅ Labeled data: Has article_id")
print(f"   ✅ Remaining data: Has article_id")
print(f"   ✅ Can track ANY article in dataset!")

print(f"\n🚀 NEXT STEPS:")
print(f"   1. Wait for Kompas & Detik data")
print(f"   2. Run same recovery for Kompas & Detik")
print(f"   3. Merge all labeled data")
print(f"   4. Train IndoBERT model")
print(f"   5. Predict on remaining_WITH_ID.csv files")
print(f"   6. Merge predictions → Full labeled dataset!")

print(f"\n{'='*70}")
print(f"✨ Random seed: 42 (reproducible!)")
print(f"✨ Full traceability: ENABLED!")
print(f"{'='*70}")

📊 FINAL SUMMARY & STATISTICS

📁 FILES CREATED/UPDATED:

1. cnbc_remaining_for_prediction_WITH_ID.csv
   • 4,261 articles for prediction
   • All have article_id for tracking
   • Ready for model inference

2. cnbc_remaining_article_ids.txt
   • List of 4,261 remaining IDs
   • For tracking & reproducibility

3. cnbc_cleaned_full_WITH_ID.csv (if created)
   • Full dataset: 5,060 articles
   • All with article_id
   • Master file for future use

📊 DATASET SPLIT:

┌─────────────────────────────────────────────┐
│ Full Dataset: 5,060 articles                │
├─────────────────────────────────────────────┤
│ ✅ Labeled (manual): 799 articles       │
│    → cnbc_labeled_WITH_ID.csv             │
│    → For training model                   │
├─────────────────────────────────────────────┤
│ 🔮 Remaining (prediction): 4,261 articles │
│    → cnbc_remaining_WITH_ID.csv           │
│    → For model inference                  │
└─────────────────────────────────────────────┘

📅 DATE DISTRIBUTION:
